# Notebook 06 — Proposed Multi-Scale STGCN

Self-contained implementation for METR-LA traffic forecasting.
Input: 12 steps (60 minutes); output: 3 steps (15 minutes); 207 sensors.

IMPORTANT:
- This is a self-contained implementation of the documented architecture.
- It may not reproduce earlier metrics unless it matches the original experiment code.
- Training overwrites /content/experiments/proposed/test_predictions.npy and test_targets.npy.
  Back up prior results if you need to preserve them.

In [ ]:
from pathlib import Path
import pickle
import json
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_STEPS = 12
OUTPUT_STEPS = 3
NUM_NODES = 207
INPUT_CHANNELS = 1
HIDDEN_CHANNELS = 32
TEMPORAL_KERNELS = [3, 5]
DROPOUT = 0.2
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 20
GRAD_CLIP = 5.0

PROJECT_ROOT = Path("/content/urbanAi")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "METR-LA"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "METR-LA"
EXPERIMENT_DIR = Path("/content/experiments/proposed")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Processed data:", PROCESSED_DIR)
print("Experiment output:", EXPERIMENT_DIR)

## 1. Load and validate processed data

In [ ]:
class TrafficDataset(Dataset):
    def __init__(self, file_path):
        data = np.load(file_path)
        self.X = torch.from_numpy(data["X"].astype(np.float32))
        self.Y = torch.from_numpy(data["Y"].astype(np.float32))
        assert self.X.ndim == 4, f"Expected 4D X, got {self.X.shape}"
        assert self.Y.ndim == 4, f"Expected 4D Y, got {self.Y.shape}"
        assert self.X.shape[1:] == (12, 207, 1), self.X.shape
        assert self.Y.shape[1:] == (3, 207, 1), self.Y.shape
        assert len(self.X) == len(self.Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.Y[index]


train_dataset = TrafficDataset(PROCESSED_DIR / "train.npz")
val_dataset = TrafficDataset(PROCESSED_DIR / "val.npz")
test_dataset = TrafficDataset(PROCESSED_DIR / "test.npz")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Input shape:", tuple(train_dataset.X.shape))
print("Target shape:", tuple(train_dataset.Y.shape))

## 2. Load and row-normalize the adjacency matrix

Confirm sensor ordering matches preprocessing if the graph was built separately.

In [ ]:
adjacency_path = RAW_DIR / "adj_METR-LA" / "adj_METR-LA.pkl"
with open(adjacency_path, "rb") as f:
    adjacency_data = pickle.load(f, encoding="latin1")

if isinstance(adjacency_data, (tuple, list)):
    adjacency = np.asarray(adjacency_data[-1], dtype=np.float32)
else:
    adjacency = np.asarray(adjacency_data, dtype=np.float32)

assert adjacency.shape == (NUM_NODES, NUM_NODES), (
    f"Expected {(NUM_NODES, NUM_NODES)}, got {adjacency.shape}"
)
degree = adjacency.sum(axis=1)
assert np.all(degree > 0), "At least one sensor has zero degree."
adjacency = adjacency / degree[:, None]
assert np.allclose(adjacency.sum(axis=1), 1.0, atol=1e-5)
A = torch.tensor(adjacency, dtype=torch.float32, device=DEVICE)

print("Adjacency shape:", tuple(A.shape))
print("First five normalized row sums:", A.sum(dim=1)[:5])

## 3. Multi-scale temporal convolution
Kernel sizes 3 and 5 operate over time, not over sensors.

In [ ]:
class MultiScaleTemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernels=(3, 5), dropout=0.2):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=(1, kernel),
                padding=(0, kernel // 2)
            )
            for kernel in kernels
        ])
        self.fusion = nn.Conv2d(
            out_channels * len(kernels), out_channels, kernel_size=1
        )
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # (B, T, N, C) -> (B, C, N, T)
        x = x.permute(0, 3, 2, 1)
        x = torch.cat([branch(x) for branch in self.branches], dim=1)
        x = self.dropout(self.activation(self.fusion(x)))
        return x.permute(0, 3, 2, 1)

## 4. Graph convolution and STGCN block

In [ ]:
class GraphConvolution(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.linear = nn.Linear(in_channels, out_channels)

    def forward(self, x, adjacency):
        # x: (B, T, N, C); adjacency: (N, N)
        spatial = torch.einsum("nm,btmc->btnc", adjacency, x)
        return self.linear(spatial)


class MultiScaleSTGCNBlock(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernels=(3, 5), dropout=0.2):
        super().__init__()
        self.temporal1 = MultiScaleTemporalConv(
            in_channels, hidden_channels, kernels, dropout
        )
        self.graph = GraphConvolution(hidden_channels, hidden_channels)
        self.temporal2 = MultiScaleTemporalConv(
            hidden_channels, hidden_channels, kernels, dropout
        )
        self.activation = nn.ReLU()

    def forward(self, x, adjacency):
        x = self.temporal1(x)
        x = self.activation(self.graph(x, adjacency))
        return self.temporal2(x)

## 5. Complete MultiScaleSTGCN model

In [ ]:
class MultiScaleSTGCN(nn.Module):
    def __init__(
        self, input_steps=12, output_steps=3, input_channels=1,
        hidden_channels=32, kernels=(3, 5), dropout=0.2
    ):
        super().__init__()
        self.block1 = MultiScaleSTGCNBlock(
            input_channels, hidden_channels, kernels, dropout
        )
        self.block2 = MultiScaleSTGCNBlock(
            hidden_channels, hidden_channels, kernels, dropout
        )
        self.temporal_head = nn.Linear(input_steps, output_steps)
        self.output_head = nn.Linear(hidden_channels, 1)

    def forward(self, x, adjacency):
        # Input: (B, 12, 207, 1)
        x = self.block1(x, adjacency)
        x = self.block2(x, adjacency)
        x = x.permute(0, 2, 3, 1)  # (B, N, C, T)
        x = self.temporal_head(x)   # (B, N, C, 3)
        x = x.permute(0, 3, 1, 2)  # (B, 3, N, C)
        return self.output_head(x)  # (B, 3, N, 1)

## 6. Instantiate and check dimensions

In [ ]:
model = MultiScaleSTGCN(
    input_steps=INPUT_STEPS,
    output_steps=OUTPUT_STEPS,
    input_channels=INPUT_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    kernels=tuple(TEMPORAL_KERNELS),
    dropout=DROPOUT,
).to(DEVICE)

X_sample, Y_sample = next(iter(train_loader))
X_sample = X_sample.to(DEVICE)
with torch.no_grad():
    prediction = model(X_sample, A)

print(model)
print("Input batch:", tuple(X_sample.shape))
print("Prediction:", tuple(prediction.shape))
print("Target batch:", tuple(Y_sample.shape))
assert prediction.shape == Y_sample.shape

## 7. Load training-only scaler

In [ ]:
scaler_data = np.load(PROCESSED_DIR / "scaler.npz")
scaler_mean = scaler_data["mean"].astype(np.float32).reshape(-1)
scaler_std = scaler_data["std"].astype(np.float32).reshape(-1)
assert scaler_mean.shape == (NUM_NODES,)
assert scaler_std.shape == (NUM_NODES,)
assert np.all(scaler_std > 0)
print("Training-only scaler loaded for", len(scaler_mean), "sensors.")

## 8. Train and select by validation MSE

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
best_val_loss = float("inf")
history = []
checkpoint_path = EXPERIMENT_DIR / "best_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for X_batch, Y_batch in train_loader:
        X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
        optimizer.zero_grad()
        predictions = model(X_batch, A)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
            val_losses.append(criterion(model(X_batch, A), Y_batch).item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    history.append({
        "epoch": epoch,
        "train_mse": train_loss,
        "validation_mse": val_loss
    })
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | Validation MSE: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": {
                "input_steps": INPUT_STEPS,
                "output_steps": OUTPUT_STEPS,
                "num_nodes": NUM_NODES,
                "hidden_channels": HIDDEN_CHANNELS,
                "kernels": TEMPORAL_KERNELS,
                "dropout": DROPOUT,
                "seed": SEED
            },
            "best_validation_mse": best_val_loss,
            "epoch": epoch
        }, checkpoint_path)

print("Training complete.")
print("Best validation MSE:", best_val_loss)
print("Checkpoint:", checkpoint_path)

## 9. Reload best checkpoint

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded best checkpoint from epoch:", checkpoint["epoch"])
print("Best validation MSE:", checkpoint["best_validation_mse"])

## 10. Predict on test set and save files for Notebook 09

Files are saved to the paths already used by the Error Analysis notebook.

In [ ]:
all_predictions = []
all_targets = []
model.eval()

with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        all_predictions.append(model(X_batch, A).cpu().numpy())
        all_targets.append(Y_batch.numpy())

predictions_scaled = np.concatenate(all_predictions, axis=0)
targets_scaled = np.concatenate(all_targets, axis=0)
assert predictions_scaled.shape == targets_scaled.shape
assert predictions_scaled.shape[1:] == (3, 207, 1)

predictions_original = (
    predictions_scaled * scaler_std.reshape(1, 1, 207, 1)
    + scaler_mean.reshape(1, 1, 207, 1)
)
targets_original = (
    targets_scaled * scaler_std.reshape(1, 1, 207, 1)
    + scaler_mean.reshape(1, 1, 207, 1)
)

np.save(
    EXPERIMENT_DIR / "test_predictions.npy",
    predictions_original.astype(np.float32)
)
np.save(
    EXPERIMENT_DIR / "test_targets.npy",
    targets_original.astype(np.float32)
)

print("Saved predictions:", predictions_original.shape)
print("Saved targets:", targets_original.shape)
print("Prediction path:", EXPERIMENT_DIR / "test_predictions.npy")
print("Target path:", EXPERIMENT_DIR / "test_targets.npy")

## 11. Overall test metrics (original scale)

In [ ]:
def calculate_metrics(y_true, y_pred, epsilon=1e-5):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = np.abs(y_true) > epsilon
    mape = (
        np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
        if np.any(mask) else float("nan")
    )
    return {"MAE": float(mae), "RMSE": float(rmse), "MAPE": float(mape)}


overall_metrics = calculate_metrics(targets_original, predictions_original)
print("Multi-Scale STGCN — Test Results")
for metric, value in overall_metrics.items():
    print(f"{metric}: {value:.4f}")

## 12. Metrics by forecast horizon

In [ ]:
horizon_metrics = {}
for horizon_index in range(OUTPUT_STEPS):
    metrics = calculate_metrics(
        targets_original[:, horizon_index],
        predictions_original[:, horizon_index]
    )
    minutes = (horizon_index + 1) * 5
    horizon_metrics[f"{minutes}_min"] = metrics
    print(f"\nHorizon: {minutes} minutes")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

## 13. Save experiment metadata and training history

In [ ]:
metadata = {
    "model_name": "MultiScaleSTGCN",
    "dataset": "METR-LA",
    "num_nodes": NUM_NODES,
    "input_steps": INPUT_STEPS,
    "output_steps": OUTPUT_STEPS,
    "hidden_channels": HIDDEN_CHANNELS,
    "temporal_kernels": TEMPORAL_KERNELS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs_requested": EPOCHS,
    "best_epoch": int(checkpoint["epoch"]),
    "best_validation_mse": float(checkpoint["best_validation_mse"]),
    "test_metrics": overall_metrics,
    "horizon_metrics": horizon_metrics,
    "prediction_shape": list(predictions_original.shape),
    "target_shape": list(targets_original.shape),
    "seed": SEED
}
with open(EXPERIMENT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
with open(EXPERIMENT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print("Saved experiment metadata and training history.")

## 14. Final verification for Notebook 09

In [ ]:
prediction_path = Path("/content/experiments/proposed/test_predictions.npy")
target_path = Path("/content/experiments/proposed/test_targets.npy")
assert prediction_path.exists()
assert target_path.exists()

saved_predictions = np.load(prediction_path)
saved_targets = np.load(target_path)
assert saved_predictions.shape == saved_targets.shape
assert saved_predictions.shape[1:] == (3, 207, 1)
assert np.isfinite(saved_predictions).all()
assert np.isfinite(saved_targets).all()

print("Verification successful.")
print("Model: MultiScaleSTGCN")
print("Predictions:", saved_predictions.shape)
print("Targets:", saved_targets.shape)
print("Notebook 09 can load these files.")